# Starter Notebook: Local Windows / Colab LoRA for Text-to-SVG

This notebook is adapted to run both on **Google Colab** and on a **local Windows PC with an NVIDIA GPU**.

What changed for local use:
- removes hard-coded `/content/...` paths
- uses local relative folders by default
- picks safer defaults based on detected GPU memory
- falls back from 8-bit optimizer if `bitsandbytes` is unavailable
- avoids the unstable Unsloth fast-inference path on Windows/Qwen3 by default
- retries generation with `use_cache=False` if rotary / KV-cache shape errors appear


## Local / Colab setup notes

### Required files
- `train.csv`: columns `id,prompt,svg`
- `test.csv`: columns `id,prompt`
- `sample_submission.csv`: optional, only if you want to compare format manually

### Local Windows recommendations
1. Use a **CUDA-enabled PyTorch** environment.
2. Put `train.csv` and `test.csv` either:
   - in the same folder as this notebook, or
   - in the folder set by `CONFIG["data_dir"]`
3. Start with the conservative defaults in `CONFIG`.
4. If inference crashes with a rotary / KV-cache mismatch, this notebook already falls back to a safer generation path.

### Practical note
This notebook is still designed for GPU fine-tuning. CPU-only execution is technically possible in some environments, but it is usually too slow for practical training with a 1.7B model.


In [2]:
# Install in Colab or in a fresh local environment if needed.
# Colab:
# %pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml
#
# Local Windows (PowerShell or notebook terminal), if your environment is missing packages:
# pip install unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml


In [3]:
import importlib.util
import os
import platform
import re
import time
import random
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

IS_WINDOWS = platform.system().lower().startswith("windows")
HAS_CUDA = torch.cuda.is_available()
HAS_BF16 = HAS_CUDA and torch.cuda.is_bf16_supported()
HAS_BNB = importlib.util.find_spec("bitsandbytes") is not None

if HAS_CUDA:
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
else:
    GPU_NAME = None
    GPU_MEM_GB = 0.0

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if HAS_CUDA:
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {HAS_CUDA}")
print(f"Running in Colab: {IS_COLAB}")
print(f"Running on Windows: {IS_WINDOWS}")
print(f"bitsandbytes available: {HAS_BNB}")
if HAS_CUDA:
    print(f"GPU: {GPU_NAME}")
    print(f"GPU memory (GB): {GPU_MEM_GB}")


c:\Users\edwar\anaconda3\envs\aidc\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.10.0+cu130
CUDA available: True
Running in Colab: False
Running on Windows: True
bitsandbytes available: True
GPU: NVIDIA GeForce RTX 4090
GPU memory (GB): 23.99


In [4]:

# Safer defaults for local Windows + Colab.
# These defaults prioritize *quality* a bit more than v2, because SVG targets are often long.


default_max_seq_length = 4096
default_train_samples = 50000
default_eval_samples = 2500
default_train_bs = 1
default_grad_accum = 16
default_infer_bs = 4


CONFIG = {
    "data_dir": ".",
    "train_csv_name": "train.csv",
    "test_csv_name": "test.csv",
    "model_name": "unsloth/Qwen3-1.7B",
    # Smaller fallback if local VRAM is tight:
    # "model_name": "unsloth/Qwen3-0.6B",
    "max_seq_length": 8192,
    "prompt_max_chars": 400,
    "max_train_samples": default_train_samples,
    "max_eval_samples": default_eval_samples,
    "eval_size": 0.02,
    "lora_r": 16,
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 2,
    "per_device_train_batch_size": default_train_bs,
    "gradient_accumulation_steps": default_grad_accum,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "inference_batch_size": default_infer_bs,
    # 256 was too short for many SVGs and caused invalid / truncated generations.
    "max_new_tokens": 8192,
    "output_dir": str(Path("outputs") / "qwen17b_svg_lora"),
    "load_in_4bit": HAS_CUDA,
    "optimizer": "paged_adamw_8bit" if (HAS_CUDA and HAS_BNB) else "adamw_torch",
    # Unsloth fast inference is useful, but Qwen3 rotary / cache errors have shown up on some Windows setups.
    "use_fast_inference": False if IS_WINDOWS else True,
    "force_no_cache_inference": True if IS_WINDOWS else False,
    # Lean prompt format reduces overhead and leaves more room for long SVG targets.
    "use_chat_format": False,
}

CONFIG


{'data_dir': '.',
 'train_csv_name': 'train.csv',
 'test_csv_name': 'test.csv',
 'model_name': 'unsloth/Qwen3-1.7B',
 'max_seq_length': 8192,
 'prompt_max_chars': 400,
 'max_train_samples': 50000,
 'max_eval_samples': 2500,
 'eval_size': 0.02,
 'lora_r': 16,
 'lora_alpha': 16,
 'learning_rate': 0.0002,
 'num_train_epochs': 2,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 16,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'inference_batch_size': 4,
 'max_new_tokens': 8192,
 'output_dir': 'outputs\\qwen17b_svg_lora',
 'load_in_4bit': True,
 'optimizer': 'paged_adamw_8bit',
 'use_fast_inference': False,
 'force_no_cache_inference': True,
 'use_chat_format': False}

In [5]:
TRAIN_CSV_PATH = 'content/train.csv'
TEST_CSV_PATH = 'content/test.csv'

print(f"Resolved train CSV: {TRAIN_CSV_PATH}")
print(f"Resolved test CSV: {TEST_CSV_PATH}")

Resolved train CSV: content/train.csv
Resolved test CSV: content/test.csv


In [6]:
# Local CSV only: the course rules do not allow external datasets.
REQUIRED_TRAIN_COLUMNS = ["id", "prompt", "svg"]
REQUIRED_TEST_COLUMNS = ["id", "prompt"]

In [7]:
def clean_prompt(text, max_chars):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:max_chars]


def is_valid_svg(svg_text):
    if not svg_text or not isinstance(svg_text, str):
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def load_local_train_df(path, sample_size=None):
    df = pd.read_csv(path)
    missing = sorted(set(REQUIRED_TRAIN_COLUMNS) - set(df.columns))
    if missing:
        raise ValueError(f"Missing train columns: {missing}")

    df = df[REQUIRED_TRAIN_COLUMNS].copy()
    df["prompt"] = df["prompt"].map(lambda x: clean_prompt(x, CONFIG["prompt_max_chars"]))
    df["svg"] = df["svg"].fillna("").astype(str).str.strip()
    df = df[df["prompt"].ne("")]
    df = df[df["svg"].map(is_valid_svg)]

    if sample_size:
        df = df.sample(n=min(sample_size, len(df)), random_state=SEED)

    return df.reset_index(drop=True)


def load_local_test_df(path):
    df = pd.read_csv(path)
    missing = sorted(set(REQUIRED_TEST_COLUMNS) - set(df.columns))
    if missing:
        raise ValueError(f"Missing test columns: {missing}")

    df = df[REQUIRED_TEST_COLUMNS].copy()
    df["prompt"] = df["prompt"].map(lambda x: clean_prompt(x, CONFIG["prompt_max_chars"]))
    df = df[df["prompt"].ne("")]
    return df.reset_index(drop=True)

In [8]:
train_df = load_local_train_df(
    TRAIN_CSV_PATH,
    sample_size=CONFIG["max_train_samples"],
)
test_df = load_local_test_df(TEST_CSV_PATH)

train_raw = Dataset.from_pandas(train_df, preserve_index=False)
splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

if CONFIG["max_eval_samples"]:
    eval_ds = eval_ds.select(range(min(CONFIG["max_eval_samples"], len(eval_ds))))

print(f"Clean train rows before split: {len(train_df)}")
print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
print(f"Test prompt rows: {len(test_df)}")
train_ds[0]

Clean train rows before split: 50000
Train rows: 49000
Eval rows: 1000
Test prompt rows: 1000


{'id': 'cda6a643-730b-4c9f-9a6c-b7fcd0674fec',
 'prompt': 'A simple black and white line drawing of a fish with stripes and a dot.',
 'svg': '<svg fill="none" height="128" viewBox="0 0 400 400" width="128" xmlns="http://www.w3.org/2000/svg"><g stroke="#000000" stroke-linecap="round" stroke-linejoin="round" stroke-opacity=".9" stroke-width="16"><path d="m72.61 231.15c62.17-37.65 241.95-151.16 262.08-9.35 4.46 31.45-40.29 33.57-59.85 39.47-84.13 25.41-202.1-78.28-206.36-76.85-7.45 2.5-2.36 26.82-1.03 32.19"/><path d="m213.15 141.45c-21.11-16.91-21.11 2.97-15.78 19.07"/><path d="m231.9 280.84c-13.96 3.61-19.52 2.6-16.7-7.5"/><path d="m296.4 195.22c-.6-.3-1.2-.6-1.8-.9"/><path d="m249.3 182.01c26.01 27.7-13.51 33.2 12.05 52.38"/><path d="m215.89 196.78c-4.44 12.68 11.2 11.79 14.66 18.6.71 1.41-31.77 9.12-6.28 21.7"/><path d="m175.42 198.13c-3.76 10.3 2.75 29.71 9.61 33.57"/></g></svg>'}

In [9]:
from unsloth import FastLanguageModel

SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup for simple icon-like images. "
    "Return only SVG code with one root <svg> element and no markdown fences."
)


def make_messages(prompt, svg=None):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    if svg is not None:
        messages.append({"role": "assistant", "content": svg})
    return messages


MANUAL_IM_START = "<|im_start|>"
MANUAL_IM_END = "<|im_end|>"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=CONFIG["load_in_4bit"],
)

def render_chat(messages, add_generation_prompt=False):
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
            )
        except TypeError:
            text = tokenizer.apply_chat_template(messages, tokenize=False)
            if add_generation_prompt and messages[-1]["role"] != "assistant":
                text += f"\n{MANUAL_IM_START}assistant\n"
            return text

    rendered = []
    for msg in messages:
        rendered.append(f"{MANUAL_IM_START}{msg['role']}\n{msg['content']}{MANUAL_IM_END}")
    if add_generation_prompt:
        rendered.append(f"{MANUAL_IM_START}assistant\n")
    return "\n".join(rendered)


def render_plain_instruction(prompt, svg=None, add_generation_prompt=False):
    text = (
        f"{SYSTEM_PROMPT}\n\n"
        f"User request:\n{prompt}\n\n"
        f"SVG:\n"
    )
    if svg is not None:
        text += svg
    elif not add_generation_prompt:
        text += ""
    return text


def build_train_text(prompt, svg):
    if CONFIG["use_chat_format"]:
        text = render_chat(
            make_messages(prompt, svg),
            add_generation_prompt=False,
        )
    else:
        text = render_plain_instruction(prompt, svg, add_generation_prompt=False)

    if tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
        text += tokenizer.eos_token
    return text


def build_infer_prompt(prompt):
    if CONFIG["use_chat_format"]:
        return render_chat(make_messages(prompt), add_generation_prompt=True)
    return render_plain_instruction(prompt, svg=None, add_generation_prompt=True)


def format_sft_text(example):
    return {"text": build_train_text(example["prompt"], example["svg"])}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:700])


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0329 17:27:55.072000 36392 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.988 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 715.87it/s]


unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Map: 100%|██████████| 1000/1000 [00:00<00:00, 23525.97 examples/s]

You generate compact, valid SVG markup for simple icon-like images. Return only SVG code with one root <svg> element and no markdown fences.

User request:
A simple black and white line drawing of a fish with stripes and a dot.

SVG:
<svg fill="none" height="128" viewBox="0 0 400 400" width="128" xmlns="http://www.w3.org/2000/svg"><g stroke="#000000" stroke-linecap="round" stroke-linejoin="round" stroke-opacity=".9" stroke-width="16"><path d="m72.61 231.15c62.17-37.65 241.95-151.16 262.08-9.35 4.46 31.45-40.29 33.57-59.85 39.47-84.13 25.41-202.1-78.28-206.36-76.85-7.45 2.5-2.36 26.82-1.03 32.19"/><path d="m213.15 141.45c-21.11-16.91-21.11 2.97-15.78 19.07"/><path d="m231.9 280.84c-13.96 3.61


In [10]:

# Quick sanity check: long SVG targets need enough context and enough generation budget.
sample_n = min(1000, len(train_df))
length_probe = train_df.sample(n=sample_n, random_state=SEED).copy()

probe_texts = [build_train_text(p, s) for p, s in zip(length_probe["prompt"], length_probe["svg"])]
token_lens = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in probe_texts]
svg_only_lens = [len(tokenizer(s, add_special_tokens=False)["input_ids"]) for s in length_probe["svg"].tolist()]

def pct(values, q):
    if not values:
        return 0
    arr = np.array(values)
    return int(np.percentile(arr, q))

print("Sampled examples:", sample_n)
print("Train text token lengths:")
print("  p50:", pct(token_lens, 50), "p90:", pct(token_lens, 90), "p95:", pct(token_lens, 95), "p99:", pct(token_lens, 99), "max:", max(token_lens))
print("SVG-only token lengths:")
print("  p50:", pct(svg_only_lens, 50), "p90:", pct(svg_only_lens, 90), "p95:", pct(svg_only_lens, 95), "p99:", pct(svg_only_lens, 99), "max:", max(svg_only_lens))
print("CONFIG max_seq_length:", CONFIG["max_seq_length"])
print("CONFIG max_new_tokens:", CONFIG["max_new_tokens"])

if pct(token_lens, 95) > CONFIG["max_seq_length"]:
    print("WARNING: p95 training examples are longer than max_seq_length. Expect truncation and weaker outputs.")
if pct(svg_only_lens, 95) > CONFIG["max_new_tokens"]:
    print("WARNING: p95 SVG targets are longer than max_new_tokens. Expect truncated / invalid generations.")


Sampled examples: 1000
Train text token lengths:
  p50: 1984 p90: 4911 p95: 5802 p99: 7178 max: 7758
SVG-only token lengths:
  p50: 1935 p90: 4858 p95: 5738 p99: 7102 max: 7708
CONFIG max_seq_length: 8192
CONFIG max_new_tokens: 8192


In [11]:
from unsloth import FastLanguageModel

if not HAS_CUDA:
    raise RuntimeError(
        "This notebook is configured for GPU fine-tuning. "
        "Please run it in a CUDA-enabled environment, or switch to a much smaller CPU-friendly workflow."
    )

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=CONFIG["load_in_4bit"],
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"  # right padding for SFT training

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print(f"Loaded model: {CONFIG['model_name']}")
print(f"4-bit loading enabled: {CONFIG['load_in_4bit']}")



==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.988 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 683.49it/s]


unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Loaded model: unsloth/Qwen3-1.7B
4-bit loading enabled: True


In [19]:
from transformers import TrainingArguments
from trl import SFTTrainer

use_bf16 = HAS_BF16
use_fp16 = HAS_CUDA and not use_bf16

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=use_fp16,
    bf16=use_bf16,
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim=CONFIG["optimizer"],
    lr_scheduler_type="cosine",
    seed=SEED,
    dataloader_num_workers=0 if IS_WINDOWS else 2,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    # Packing can slightly blur chat-turn boundaries for short instruction data.
    # Keep each prompt/response pair separate for more faithful instruction tuning.
    packing=False,
    args=training_args,
)

train_result = trainer.train()
train_result



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Unsloth: Tokenizing ["text"]: 100%|██████████| 1000/1000 [00:02<00:00, 474.93 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49,000 | Num Epochs = 2 | Total steps = 6,126
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)


Step,Training Loss,Validation Loss
100,0.609267,0.575834
200,0.516076,0.476849
300,0.445596,0.442868
400,0.430488,0.421634
500,0.397457,0.407303
600,0.401206,0.399192
700,0.369748,0.389093
800,0.376739,0.381896
900,0.360184,0.375381
1000,0.369858,0.371453


c:\Users\edwar\anaconda3\envs\aidc\lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\edwar\anaconda3\envs\aidc\lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\edwar\anaconda3\envs\aidc\lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Trans

TrainOutput(global_step=6126, training_loss=0.35096691561959514, metrics={'train_runtime': 21987.3389, 'train_samples_per_second': 4.457, 'train_steps_per_second': 0.279, 'total_flos': 7.642224855153746e+17, 'train_loss': 0.35096691561959514, 'epoch': 2.0})

In [20]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

Saved adapter + tokenizer to: outputs\qwen17b_svg_lora


In [12]:
# Load the trained model for inference.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["output_dir"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None
)

# Safer local inference path.
if CONFIG["use_fast_inference"]:
    FastLanguageModel.for_inference(model)
else:
    print("Skipping FastLanguageModel.for_inference() for local stability.")

model.eval()
tokenizer.padding_side = "left"  # left padding is safer for batched causal generation

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)


def extract_svg(text):
    if not text:
        return ""

    # Remove markdown fences if the model emits them.
    text = text.replace("```svg", "```").replace("```XML", "```").replace("```xml", "```")
    text = text.replace("```", "").strip()

    # Best case: a full SVG block is already present.
    match = SVG_REGEX.search(text)
    if match:
        return match.group(0).strip()

    # Partial recovery: if we see <svg but no closing tag, append it and try again.
    start = text.lower().find("<svg")
    if start != -1:
        candidate = text[start:].strip()
        if "</svg>" not in candidate.lower():
            candidate += "</svg>"
        match = SVG_REGEX.search(candidate)
        if match:
            return match.group(0).strip()

    return ""


_GENERIC_FALLBACK = (
    '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
    '<rect x="0" y="0" width="256" height="256" fill="white"/>'
    '<circle cx="128" cy="128" r="64" fill="black"/>'
    '</svg>'
)


def prompt_tokens_for_retrieval(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))


# Retrieval fallback is much better than returning the same dummy circle for every failed row.
_train_prompt_token_sets = [prompt_tokens_for_retrieval(p) for p in train_df["prompt"].tolist()]
_train_svgs = train_df["svg"].tolist()


def nearest_train_svg(prompt, min_score=0.08):
    query = prompt_tokens_for_retrieval(prompt)
    if not query:
        return _GENERIC_FALLBACK

    best_idx = -1
    best_score = -1.0
    for idx, toks in enumerate(_train_prompt_token_sets):
        union = len(query | toks) or 1
        overlap = len(query & toks)
        score = overlap / union
        if score > best_score:
            best_score = score
            best_idx = idx

    if best_idx >= 0 and best_score >= min_score:
        return _train_svgs[best_idx]
    return _GENERIC_FALLBACK


def fallback_svg(prompt):
    return nearest_train_svg(prompt)


def _decode_generated_only(output_ids, attention_mask):
    generated_texts = []
    input_lengths = attention_mask.sum(dim=1).tolist()
    for row_ids, input_len in zip(output_ids, input_lengths):
        new_token_ids = row_ids[int(input_len):]
        text = tokenizer.decode(new_token_ids, skip_special_tokens=True)
        generated_texts.append(text)
    return generated_texts


def _generate_once(prompts, max_new_tokens, use_cache):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    generate_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.02,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=use_cache,
    )

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **generate_kwargs,
        )
    return _decode_generated_only(output_ids, inputs["attention_mask"])


def generate_svg_batch(prompts, max_new_tokens=None):
    # max_new_tokens = max_new_tokens or CONFIG["max_new_tokens"]
    max_new_tokens = 4096
    infer_prompts = [build_infer_prompt(prompt) for prompt in prompts]

    preferred_use_cache = not CONFIG["force_no_cache_inference"]

    try:
        decoded = _generate_once(
            infer_prompts,
            max_new_tokens=max_new_tokens,
            use_cache=preferred_use_cache,
        )
    except RuntimeError as e:
        error_text = str(e)
        shape_mismatch = (
            "broadcast shape" in error_text
            or "doesn't match the broadcast shape" in error_text
        )
        if not shape_mismatch:
            raise

        print("Generation hit a rotary/KV-cache shape mismatch. Retrying one prompt at a time with use_cache=False.")
        decoded = []
        for prompt in prompts:
            single_infer_prompt = [build_infer_prompt(prompt)]
            decoded.extend(
                _generate_once(
                    single_infer_prompt,
                    max_new_tokens=max_new_tokens,
                    use_cache=False,
                )
            )

    svgs = []
    for prompt, text in zip(prompts, decoded):
        svg = extract_svg(text)
        if not is_valid_svg(svg):
            svg = fallback_svg(prompt)
        svgs.append(svg)
    return svgs


test_prompt = "a simple blue bird icon"

pred_svg = generate_svg_batch([test_prompt])[0]
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))


==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.988 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 696.22it/s]


unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Skipping FastLanguageModel.for_inference() for local stability.


Both `max_new_tokens` (=4096) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\edwar\anaconda3\envs\aidc\lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\edwar\anaconda3\envs\aidc\lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning

KeyboardInterrupt: 

In [13]:
# Local-Windows-friendly inference config
CONFIG.setdefault("infer_load_max_seq_length", 1024)   # do NOT reuse huge training seq length here
CONFIG.setdefault("infer_prompt_max_length", 256)      # prompts are short; keep them short at inference
CONFIG.setdefault("infer_max_new_tokens", 1536)        # start conservative locally
CONFIG.setdefault("use_fast_inference", False)         # safer on Windows / local
CONFIG.setdefault("force_no_cache_inference", True)    # avoids rotary / KV-cache mismatch + lowers VRAM spikes

# Load the trained model for inference with a smaller local inference context
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["output_dir"],
    max_seq_length=CONFIG["infer_load_max_seq_length"],
    dtype=None,
)

if CONFIG["use_fast_inference"]:
    FastLanguageModel.for_inference(model)
else:
    print("Skipping FastLanguageModel.for_inference() for local stability.")

model.eval()
tokenizer.padding_side = "left"

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

def extract_svg(text):
    if not text:
        return ""

    text = text.replace("```svg", "```").replace("```XML", "```").replace("```xml", "```")
    text = text.replace("```", "").strip()

    match = SVG_REGEX.search(text)
    if match:
        return match.group(0).strip()

    start = text.lower().find("<svg")
    if start != -1:
        candidate = text[start:].strip()
        if "</svg>" not in candidate.lower():
            candidate += "</svg>"
        match = SVG_REGEX.search(candidate)
        if match:
            return match.group(0).strip()

    return ""

_GENERIC_FALLBACK = (
    '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
    '<rect x="0" y="0" width="256" height="256" fill="white"/>'
    '<circle cx="128" cy="128" r="64" fill="black"/>'
    '</svg>'
)

def prompt_tokens_for_retrieval(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))

_train_prompt_token_sets = [prompt_tokens_for_retrieval(p) for p in train_df["prompt"].tolist()]
_train_svgs = train_df["svg"].tolist()

def nearest_train_svg(prompt, min_score=0.08):
    query = prompt_tokens_for_retrieval(prompt)
    if not query:
        return _GENERIC_FALLBACK

    best_idx = -1
    best_score = -1.0
    for idx, toks in enumerate(_train_prompt_token_sets):
        union = len(query | toks) or 1
        overlap = len(query & toks)
        score = overlap / union
        if score > best_score:
            best_score = score
            best_idx = idx

    if best_idx >= 0 and best_score >= min_score:
        return _train_svgs[best_idx]
    return _GENERIC_FALLBACK

def fallback_svg(prompt):
    return nearest_train_svg(prompt)

def _decode_generated_only(output_ids, attention_mask):
    generated_texts = []
    input_lengths = attention_mask.sum(dim=1).tolist()
    for row_ids, input_len in zip(output_ids, input_lengths):
        new_token_ids = row_ids[int(input_len):]
        text = tokenizer.decode(new_token_ids, skip_special_tokens=True)
        generated_texts.append(text)
    return generated_texts

def _generate_once(prompts, max_new_tokens, use_cache):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONFIG["infer_prompt_max_length"],
    ).to(model.device)

    # Avoid the "both max_new_tokens and max_length" warning + huge VRAM reservations
    input_len = inputs["input_ids"].shape[1]
    max_length = min(
        CONFIG["infer_load_max_seq_length"],
        input_len + max_new_tokens,
    )

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=max_length,
            do_sample=False,
            repetition_penalty=1.02,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=use_cache,
        )

    return _decode_generated_only(output_ids, inputs["attention_mask"])

def generate_svg_batch(prompts, max_new_tokens=None):
    max_new_tokens = max_new_tokens or CONFIG["infer_max_new_tokens"]

    svgs = []
    for prompt in prompts:
        infer_prompt = build_infer_prompt(prompt)
        preferred_use_cache = not CONFIG["force_no_cache_inference"]

        try:
            decoded = _generate_once(
                [infer_prompt],
                max_new_tokens=max_new_tokens,
                use_cache=preferred_use_cache,
            )[0]
        except RuntimeError as e:
            error_text = str(e)
            shape_mismatch = (
                "broadcast shape" in error_text
                or "doesn't match the broadcast shape" in error_text
            )
            oom = "out of memory" in error_text.lower()

            if not (shape_mismatch or oom):
                raise

            print("Retrying single-prompt generation with use_cache=False and a smaller decode budget.")
            safer_tokens = min(max_new_tokens, 1024)
            decoded = _generate_once(
                [infer_prompt],
                max_new_tokens=safer_tokens,
                use_cache=False,
            )[0]

        svg = extract_svg(decoded)
        if not is_valid_svg(svg):
            svg = fallback_svg(prompt)
        svgs.append(svg)

    return svgs

# Quick test
test_prompt = "a simple blue bird icon"
pred_svg = generate_svg_batch([test_prompt])[0]
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))

==((====))==  Unsloth 2026.3.17: Fast Qwen3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.988 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 716.69it/s]


unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Skipping FastLanguageModel.for_inference() for local stability.
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#E0EBFE" fill-opacity="1.0"  filling="0" d="M132.43800354003906 67.625 L67.56199645996094 132.375 L25.938003540039062 90.75 C8.0 72.81199645996094 8.0 43.8120002746582 25.938003540039062 25.938003540039062 A45.76300048828125 45.76300048828125 0.0 0 1 58.375 12.5 C70.06199645996094 12.5 81.81199645996094 17.0 90.75 25.938003540039062 L132.43800354003906 67.625 Z"></path>
<path fill="
Valid SVG: True


In [14]:
print(pred_svg)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#E0EBFE" fill-opacity="1.0"  filling="0" d="M132.43800354003906 67.625 L67.56199645996094 132.375 L25.938003540039062 90.75 C8.0 72.81199645996094 8.0 43.8120002746582 25.938003540039062 25.938003540039062 A45.76300048828125 45.76300048828125 0.0 0 1 58.375 12.5 C70.06199645996094 12.5 81.81199645996094 17.0 90.75 25.938003540039062 L132.43800354003906 67.625 Z"></path>
<path fill="#5465CF" fill-opacity="1.0"  filling="0" d="M53.875 31.811996459960938 A3.1059999465942383 3.1059999465942383 0.0 0 0 54.45000076293945 31.755996704101562 C56.25 31.41899871826172 58.09400177001953 31.25 59.9379997253418 31.25 A3.125 3.125 0.0 1 0 59.9379997253418 25.0 C57.70600128173828 25.0 55.474998474121094 25.20600128173828 53.29999923706055 25.619003295898438 A3.125 3.125 0.0 0 0 53.86899948120117 31.805999755859375 L53.875 31.811996459960938 Z M35.23100280761719 84.64399719238281 L55.2309

In [ ]:

SUBMISSION_PATH = "./submission.csv"

rows = []
fallback_count = 0
t0 = time.time()

for start in range(0, len(test_df), CONFIG["inference_batch_size"]):
    print(f"Processing rows {start} to {min(start + CONFIG['inference_batch_size'], len(test_df))}...")
    batch_df = test_df.iloc[start : start + CONFIG["inference_batch_size"]]
    batch_prompts = batch_df["prompt"].tolist()
    batch_svgs = generate_svg_batch(batch_prompts)

    for sample_id, prompt, svg in zip(batch_df["id"], batch_prompts, batch_svgs):
        if not is_valid_svg(svg):
            fallback_count += 1
            svg = fallback_svg(prompt)
        rows.append({"id": sample_id, "svg": svg})

sub_df = pd.DataFrame(rows)
assert list(sub_df.columns) == ["id", "svg"]
assert len(sub_df) == len(test_df)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Fallback count: {fallback_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()


Processing rows 0 to 4...


c:\Users\edwar\anaconda3\envs\aidc\lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
c:\Users\edwar\anaconda3\envs\aidc\lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Processing rows 4 to 8...
Processing rows 8 to 12...
Processing rows 12 to 16...
Processing rows 16 to 20...
Processing rows 20 to 24...
Processing rows 24 to 28...
Processing rows 28 to 32...
Processing rows 32 to 36...
Processing rows 36 to 40...
Processing rows 40 to 44...
Processing rows 44 to 48...
Processing rows 48 to 52...
Processing rows 52 to 56...
Processing rows 56 to 60...
Processing rows 60 to 64...
Processing rows 64 to 68...
Processing rows 68 to 72...
Processing rows 72 to 76...
Processing rows 76 to 80...
Processing rows 80 to 84...
Processing rows 84 to 88...
Processing rows 88 to 92...
Processing rows 92 to 96...
Processing rows 96 to 100...
Processing rows 100 to 104...
Processing rows 104 to 108...
Processing rows 108 to 112...
Processing rows 112 to 116...
Processing rows 116 to 120...
Processing rows 120 to 124...
Processing rows 124 to 128...
Processing rows 128 to 132...
Processing rows 132 to 136...
Processing rows 136 to 140...
Processing rows 140 to 144...


In [ ]:
# Save / export helpers for both Colab and local runs.
ARTIFACT_ZIP_BASE = str(Path(CONFIG["output_dir"]).with_name(Path(CONFIG["output_dir"]).name + "_artifacts"))

import shutil

if Path(CONFIG["output_dir"]).exists():
    shutil.make_archive(ARTIFACT_ZIP_BASE, "zip", CONFIG["output_dir"])
    print(f"Created artifact archive: {ARTIFACT_ZIP_BASE}.zip")

if IS_COLAB:
    from google.colab import files  # type: ignore
    print("Uncomment the lines below to download outputs to your machine.")
    # files.download(SUBMISSION_PATH)
    # files.download(ARTIFACT_ZIP_BASE + ".zip")
else:
    print(f"Submission saved at: {Path(SUBMISSION_PATH).resolve()}")
    print(f"Model artifacts saved at: {Path(CONFIG['output_dir']).resolve()}")
    print(f"Zipped artifacts saved at: {Path(ARTIFACT_ZIP_BASE + '.zip').resolve()}")


## Notes

- This notebook now defaults to **local relative paths** instead of hard-coded Colab paths.
- If local VRAM is limited, try:
  - `CONFIG["model_name"] = "unsloth/Qwen3-0.6B-Base"`
  - smaller `max_seq_length`
  - smaller `max_train_samples`
  - `inference_batch_size = 1`
- On some local Windows + Qwen3 + Unsloth combinations, batched generation can fail with a rotary / KV-cache mismatch.
  This notebook works around that by skipping fast inference on Windows and retrying generation with `use_cache=False`.
- If you later move back to Colab, you can turn fast inference back on by setting:
  `CONFIG["use_fast_inference"] = True`
